# HW3 : Evaluating Binary Classifiers



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

from sklearn.model_selection import train_test_split

RANDOM_SEED = 8675309

# Starter Code
## Data loading and cleaning

In [83]:
# You may need to adjust the path
diabetes_readmit = pd.read_csv("diabetic_readmission.csv")

# Remove target variable (biopsy) and other diagnostic tests we're not using
X_df = diabetes_readmit.drop(columns=["readmitted", "encounter_id", "patient_nbr"])

# We are only trying to predict if they were readmitted within 30 days
readmitted_mapping = {'<30': 1, '>30': 0, 'NO': 0}
diabetes_readmit.readmitted = diabetes_readmit.readmitted.map(readmitted_mapping)
y = diabetes_readmit.readmitted

In [84]:
diabetes_readmit.describe()

,Unnamed: 0,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,readmitted
count,14250.000000,1.425000e+04,1.425000e+04,14250.000000,14250.000000,14250.000000,14250.000000,14250.000000,14250.000000,14250.000000,14250.000000,14250.000000,14250.000000,14250.000000,14250.000000
mean,51079.510947,1.656705e+08,5.431133e+07,2.021544,3.732281,5.708842,4.393825,42.911298,1.345965,16.031789,0.391298,0.260211,0.773895,7.380912,0.080000
std,29205.871675,1.020231e+08,3.856413e+07,1.445343,5.348948,4.018773,3.007580,20.064738,1.701749,8.195488,1.347276,1.234892,1.638831,1.949795,0.271303
min,8.000000,1.252200e+04,3.780000e+02,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,25876.500000,8.582953e+07,2.342251e+07,1.000000,1.000000,1.000000,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000,0.000000
50%,50955.500000,1.525163e+08,4.578918e+07,1.000000,1.000000,7.000000,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000,0.000000
75%,76439.000000,2.307733e+08,8.794194e+07,3.000000,4.000000,7.000000,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000,0.000000
max,101765.000000,4.438672e+08,1.892578e+08,8.000000,28.000000,22.000000,14.000000,120.000000,6.000000,74.000000,40.000000,64.000000,21.000000,16.000000,1.000000


Missing data is represented by "?", which we need to convert to `NaN`. 

In [85]:
#Replace all of the "?" with NaN (not a number)
X_df = X_df.replace('?', np.nan)

X_df.dtypes

Unnamed: 0                   int64
race                        object
gender                      object
age                         object
weight                      object
admission_type_id            int64
discharge_disposition_id     int64
admission_source_id          int64
time_in_hospital             int64
payer_code                  object
medical_specialty           object
num_lab_procedures           int64
num_procedures               int64
num_medications              int64
number_outpatient            int64
number_emergency             int64
number_inpatient             int64
diag_1                      object
diag_2                      object
diag_3                      object
number_diagnoses             int64
max_glu_serum               object
A1Cresult                   object
metformin                   object
repaglinide                 object
nateglinide                 object
chlorpropamide              object
glimepiride                 object
acetohexamide       

We need to make an initial split so we don't make decisions based on missing data using the test set. Notice we include the `stratify` keyword, which splits the data such that there are equal proportions of each class in the training and testing sets. This is especially important for unbalanced datasets like this one. 

In [86]:
# Create a development/test split
X_dev, X_test, y_dev, y_test = train_test_split(
    X_df, y, test_size=0.10, random_state=RANDOM_SEED, stratify=y
)

### Missing Data

This dataset contains missing data. We'll deal with missing values in the pipeline, but let's get an initial feel for the percent of missing data per feature. Note that we're not using the test set to perform any missing data calculations or decisions. 

In [87]:
missing_counts = X_dev.isnull().sum()
missing_percentage = (X_dev.isnull().sum() / len(X_dev)) * 100

# Combine into a clean table
missing_info = pd.concat([missing_counts, missing_percentage], axis=1, keys=['Total Missing', 'Percent'])
print(missing_info.sort_values(by='Percent', ascending=False).head(10))

                     Total Missing    Percent
weight                       12406  96.732943
max_glu_serum                12148  94.721248
A1Cresult                    10685  83.313840
medical_specialty             6244  48.686160
payer_code                    5044  39.329435
race                           256   1.996101
diag_3                         173   1.348928
diag_2                          39   0.304094
diag_1                           2   0.015595
admission_source_id              0   0.000000


We have some features that are almost entirely missing. We'll be conservative since we have so many features and drop everything that has 80% missing data or more. Payer code and medical specialty aren't missing that much data, but as categorical features with a lot of categories they are difficult to impute, so we could consider dropping them as well.  

In [88]:
cols_to_drop = X_dev.columns[X_dev.isna().mean() > 0.80].tolist()
X_dev = X_dev.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

### Categorical data

The age column is stored in 10-year increments. We don't want to treat this as a pure categorical variable, as it would obfuscate that 20 is closer to 30 than it is to 60, for instance. Instead, let's convert age to be the mean of each interval. 

In [89]:
# Create a dictionary mapping the exact strings in the dataset to their midpoints
age_mapping = {
    '[0-10)': 5,
    '[10-20)': 15,
    '[20-30)': 25,
    '[30-40)': 35,
    '[40-50)': 45,
    '[50-60)': 55,
    '[60-70)': 65,
    '[70-80)': 75,
    '[80-90)': 85,
    '[90-100)': 95
}
X_dev.age = X_dev.age.map(age_mapping)
X_test.age = X_test.age.map(age_mapping)

Now let's grab all other categorical columns, those either marked as categories or as objects (strings) by pandas. We'll actually do the encoding inside the pipeline using sklearn's `OneHotEncoder`. There are lots of other options for dealing with the categorical data we could consider, but this is the easiest approach. 

In [90]:
categorical_cols = X_dev.select_dtypes(include=['object', 'category']).columns

Finally, we'll further split out a validation set to use for hyperparamter selection. 

In [91]:
# Further split the development set into a training and validation set
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.2, random_state=RANDOM_SEED, stratify=y_dev
)

## Pipelines

Both Logistic Regression (with regularization) and K-Nearest Neighbors require feature scaling to work correctly. These pipelines ensures that missing values are imputed and features are standardized *after* any data splitting to prevent data leakage.

In [ ]:
"""
Creates a sklearn pipeline for logistic regression with imputation and rescaling steps.
C is the inverse regularization strength, smaller values resulting in higher regularization
Passes in l1_ratio=0 for L2-regularized logistic regression and uses the lbfgs solver.
"""

# Since we only want to apply the OneHotEncoder to some columns, we have to put it in a ColumnTransformer.
categories_transformer = ColumnTransformer(
    transformers=[ 
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ], 

    # Without this line sklearn drops all columns not transformed
    remainder='passthrough')

def make_logit_knnimpute(C=1.0, k=5):
    pipeline = Pipeline(
        steps=[
         ('cat_encoder', categories_transformer),
         ('imputer', KNNImputer(n_neighbors=k)),
         ('rescaler', MinMaxScaler()),
         ('logit', LogisticRegression(solver="lbfgs", l1_ratio=0, C=C, max_iter=1000))
        ])
    
    # Return the constructed pipeline
    return pipeline

def make_knn_knnimpute(k_model=5, k_impute=5):
    pipeline = Pipeline(
        steps=[
         ('cat_encoder', categories_transformer),
         ('imputer', KNNImputer(n_neighbors=k_impute)),
         ('rescaler', MinMaxScaler()),
         ('knn', KNeighborsClassifier(n_neighbors=k_model))
        ])
    
    # Return the constructed pipeline
    return pipeline

# Your Code

Good luck!